In [2]:
import os

# uncomment to disable NVIDIA GPUs
#os.environ['CUDA_VISIBLE_DEVICES'] = ''
# or pick the device (cpu, gpu, and tpu)
#os.environ['JAX_PLATFORMS'] = 'cpu'

# change JAX GPU memory preallocation fraction
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.95'

# you do not want this
#os.environ['XLA_FLAGS'] = '--xla_gpu_deterministic_ops=true'

import jax
#jax.print_environment_info()

!nvidia-smi --query-gpu=gpu_name --format=csv,noheader

import matplotlib.pyplot as plt
import matplotlib_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('jpeg')


from pmwd import (
    Configuration,
    Cosmology, SimpleLCDM,
    boltzmann, linear_power, growth,
    white_noise, linear_modes,
    lpt,
    nbody,
    scatter,
)
from pmwd.pm_util import fftinv
from pmwd.spec_util import powspec
from pmwd.vis_util import simshow

/bin/bash: linha 1: nvidia-smi: comando não encontrado


In [3]:
if jax.default_backend() == 'gpu':
    ptcl_spacing = 1.  # Lagrangian space Cartesian particle grid spacing, in Mpc/h by default
    ptcl_grid_shape = (256,) * 3
else:
    ptcl_spacing = 4.
    ptcl_grid_shape = (128,) * 3

conf = Configuration(ptcl_spacing, ptcl_grid_shape, mesh_shape=2)  # 2x mesh shape


print(conf)  # with other default parameters
print(f'Simulating {conf.ptcl_num} particles with a {conf.mesh_shape} mesh for {conf.a_nbody_num} time steps.')

Configuration(ptcl_spacing=4.0,
              ptcl_grid_shape=(128, 128, 128),
              mesh_shape=(256, 256, 256),
              cosmo_dtype=dtype('float64'),
              pmid_dtype=dtype('int16'),
              float_dtype=dtype('float32'),
              k_pivot_Mpc=0.05,
              T_cmb=2.7255,
              M=1.98847e+40,
              L=3.0856775815e+22,
              T=3.0856775815e+17,
              transfer_fit=True,
              transfer_fit_nowiggle=False,
              transfer_lgk_min=-4,
              transfer_lgk_max=3,
              transfer_lgk_maxstep=0.0078125,
              growth_rtol=1.4901161193847656e-08,
              growth_atol=1.4901161193847656e-08,
              growth_inistep=(1, None),
              lpt_order=2,
              a_start=0.015625,
              a_stop=1,
              a_lpt_maxstep=0.0078125,
              a_nbody_maxstep=0.015625,
              symp_splits=((0, 0.5), (1, 0.5)),
              chunk_size=16777216)
Simulating 209715

In [4]:
cosmo = SimpleLCDM(conf)
cosmos1 = str(cosmo)
print(cosmos1)

Cosmology(A_s_1e9=Array(2., dtype=float64),
          n_s=Array(0.96, dtype=float64),
          Omega_m=Array(0.3, dtype=float64),
          Omega_b=Array(0.05, dtype=float64),
          h=Array(0.7, dtype=float64),
          Omega_k_=None,
          w_0_=None,
          w_a_=None,
          mu_0_=None,
          k_c_=None,
          k_analyze_=None,
          transfer=None,
          growth=None,
          growth_kgrid=None,
          varlin=None)


In [5]:
%time cosmo = jax.block_until_ready(boltzmann(cosmo, conf))
%timeit jax.block_until_ready(boltzmann(cosmo, conf))
seed = 0
modes = white_noise(seed, conf)

modes = linear_modes(modes, cosmo, conf)

%time ptcl, obsvbl = jax.block_until_ready(lpt(modes, cosmo, conf))
%timeit jax.block_until_ready(lpt(modes, cosmo, conf))
ptcl.disp.std(), ptcl.vel.std()

%time jax.block_until_ready(nbody(ptcl, obsvbl, cosmo, conf))
%time ptcl, obsvbl = jax.block_until_ready(nbody(ptcl, obsvbl, cosmo, conf))
ptcl.disp.std(), ptcl.vel.std()

CPU times: user 896 ms, sys: 65.1 ms, total: 961 ms
Wall time: 618 ms
1.2 ms ± 46.6 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
CPU times: user 632 ms, sys: 77.2 ms, total: 709 ms
Wall time: 312 ms
116 ms ± 1.79 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
CPU times: user 1min 4s, sys: 42.5 s, total: 1min 47s
Wall time: 44.5 s
CPU times: user 1min 3s, sys: 44.1 s, total: 1min 47s
Wall time: 44.2 s


(Array(6.034, dtype=float32), Array(3.306, dtype=float32))

In [10]:
import re

meta_str = str(cosmo)
cosmo1 = str(cosmo)
#print(cosmos1)

#meta_str = meta_str.replace(",\s*(?:transfer|growth|varlin)=Array\([^)]*\)", "")
meta_str = re.sub(r",\s*(?:transfer|growth|varlin)=Array\([^)]*\)", "", meta_str)

s = repr(cosmo)  # ou a string que você mostrou
s_sem = re.sub(
    r",\s*transfer=Array\(\[.*?\]\s*,\s*dtype=[^)]+\)",
    "",
    s,
    flags=re.DOTALL
)
print(meta_str)

Cosmology(A_s_1e9=Array(2., dtype=float64),
          n_s=Array(0.96, dtype=float64),
          Omega_m=Array(0.3, dtype=float64),
          Omega_b=Array(0.05, dtype=float64),
          h=Array(0.7, dtype=float64),
          Omega_k_=None,
          w_0_=None,
          w_a_=None,
          mu_0_=None,
          k_c_=None,
          k_analyze_=None,
          growth_kgrid=None)
